# Single-cell trajectory inference with OT-CFM (JAX)

JAX/Equinox port of the TorchCFM single-cell example, using [`jax-cfm`](https://github.com/nklkhlr/jax-cfm).

We have snapshots of cells at several timepoints (unpaired, since cells are destroyed when measured). We train **one** velocity field `v(t, x)` with OT-CFM across all consecutive timepoint pairs, then integrate cells from one timepoint forward and compare them to the observed next timepoint.

**Caveats about this port**
- The `jax-cfm` call is isolated in one adapter cell (`cfm_sample`). I could not inspect that package's source while writing this, so the call signature follows TorchCFM's API and may need a small tweak.
- The data loader assumes the TrajectoryNet-style `.npz` (`pcs`, `sample_labels`). If you don't have it, a synthetic dataset is used so the notebook still runs end to end.

In [1]:
# !pip install jax equinox optax diffrax ott-jax matplotlib numpy
# !pip install git+https://github.com/nklkhlr/jax-cfm

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import optax
import diffrax

from ott.geometry import pointcloud
from ott.problems.linear import linear_problem
from ott.solvers.linear import sinkhorn

In [3]:
from jax_mc_pilco.model_learning.flows.conditional_flow_matching import ExactOptimalTransportConditionalFlowMatcher

ImportError: cannot import name 'ExactOptimalTransportConditionalFlowMatcher' from 'jax_mc_pilco.model_learning.flows.conditional_flow_matching' (/Users/danielmarthaler/dev/jax-mc-pilco/src/jax_mc_pilco/model_learning/flows/conditional_flow_matching.py)

In [14]:
print(jax.devices())

[CpuDevice(id=0)]


## Data
One array per timepoint, standardized with a shared mean/std.

In [ ]:
DATA_PATH = None  # e.g. "eb_velocity_v5.npz"; None -> synthetic data
DIM = 5

def load_snapshots(path=None, dim=DIM, seed=0):
    if path is not None:
        d = np.load(path, allow_pickle=True)
        X, labels = d["pcs"][:, :dim], d["sample_labels"]
        snaps = [X[labels == l] for l in np.unique(labels)]
    else:  # synthetic: blobs drifting along a curve, 5 timepoints
        rng = np.random.default_rng(seed)
        snaps = []
        for i in range(5):
            c = np.zeros(dim)
            c[0], c[1] = 2.0 * i, np.sin(i)
            spread = 0.3 + 0.1 * i
            snaps.append(c + spread * rng.standard_normal((1500, dim)))
    allx = np.concatenate(snaps)
    mu, sd = allx.mean(0), allx.std(0)
    return [jnp.asarray((s - mu) / sd, dtype=jnp.float32) for s in snaps], (mu, sd)

snaps, _ = load_snapshots(DATA_PATH)
n_t = len(snaps)
print([s.shape for s in snaps])

fig, ax = plt.subplots(figsize=(5, 4))
for i, s in enumerate(snaps):
    ax.scatter(s[:, 0], s[:, 1], s=2, alpha=0.4, label=f"t={i}")
ax.legend(markerscale=5); ax.set_title("Observed snapshots (first 2 dims)")
plt.show()

## Model
Velocity field `v_theta(t, x)`, where `t` is the *global* time in `[0, n_t - 1]` (timepoint `i` sits at `t = i`).

In [ ]:
class VelocityNet(eqx.Module):
    mlp: eqx.nn.MLP

    def __init__(self, dim, width, key):
        self.mlp = eqx.nn.MLP(dim + 1, dim, width_size=width, depth=3,
                              activation=jax.nn.selu, key=key)

    def __call__(self, t, x):
        return self.mlp(jnp.concatenate([x, jnp.atleast_1d(t)]))

key = jr.PRNGKey(0)
key, mkey = jr.split(key)
model = VelocityNet(DIM, width=64, key=mkey)

## Flow matcher (the only `jax-cfm`-specific cell)
TorchCFM: `t, xt, ut = fm.sample_location_and_conditional_flow(x0, x1)`.
JAX has no global RNG, so this adapter passes a PRNG key. **If your installed version's signature differs, edit only this function.** Ours returns `tau in [0,1]`, `xt`, `ut` with a leading batch dimension.

In [ ]:
SIGMA = 0.1
fm = ExactOptimalTransportConditionalFlowMatcher(sigma=SIGMA)

def cfm_sample(key, x0, x1):
    tau, xt, ut = fm.sample_location_and_conditional_flow(key, x0, x1)  # <- adjust if needed
    return tau, xt, ut

## Training
Each step: pick an interval `i -> i+1`, draw minibatches from both snapshots, couple them with minibatch OT, and regress `v_theta` onto the conditional velocity. Within the interval, local time `tau` is mapped to global time `i + tau`.

In [3]:
BATCH = 256
STEPS = 20_000
opt = optax.adam(1e-3)
opt_state = opt.init(eqx.filter(model, eqx.is_array))

def loss_fn(model, key, x0, x1, i):
    tau, xt, ut = cfm_sample(key, x0, x1)
    t_glob = i + tau
    vt = jax.vmap(model)(t_glob, xt)
    return jnp.mean((vt - ut) ** 2)

@eqx.filter_jit
def train_step(model, opt_state, key, x0, x1, i):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, key, x0, x1, i)
    updates, opt_state = opt.update(grads, opt_state, model)
    return eqx.apply_updates(model, updates), opt_state, loss

def draw(key, arr, n):
    return arr[jr.randint(key, (n,), 0, arr.shape[0])]

t0 = time.time()
losses = []
for step in range(STEPS):
    key, k_i, k_a, k_b, k_fm = jr.split(key, 5)
    i = int(jr.randint(k_i, (), 0, n_t - 1))
    x0, x1 = draw(k_a, snaps[i], BATCH), draw(k_b, snaps[i + 1], BATCH)
    model, opt_state, loss = train_step(model, opt_state, k_fm, x0, x1, float(i))
    losses.append(float(loss))
    if (step + 1) % 5000 == 0:
        print(f"{step+1}: loss {np.mean(losses[-500:]):.4f}  time {time.time()-t0:.1f}s")

NameError: name 'model' is not defined

## Sampling: integrate cells forward with an ODE solver
`diffrax` replaces `torchdyn`'s `NeuralODE`. `model(t, x)` is the vector field.

In [4]:
def integrate(model, x_start, t_start, t_end, n_save=11):
    term = diffrax.ODETerm(lambda t, y, args: model(t, y))
    ts = jnp.linspace(t_start, t_end, n_save)
    def one(y0):
        sol = diffrax.diffeqsolve(term, diffrax.Tsit5(), t_start, t_end, dt0=0.05, y0=y0,
                                  saveat=diffrax.SaveAt(ts=ts),
                                  stepsize_controller=diffrax.PIDController(rtol=1e-4, atol=1e-5))
        return sol.ys
    return jax.vmap(one)(x_start)  # (n_cells, n_save, dim)

# One-step-ahead predictions: start from snapshot i, predict snapshot i+1
traj = {i: integrate(model, snaps[i], float(i), float(i + 1)) for i in range(n_t - 1)}

NameError: name 'n_t' is not defined

## Evaluation: Sinkhorn (entropic OT) cost, predicted vs observed next snapshot

In [5]:
def ot_cost(a, b, eps=0.05):
    geom = pointcloud.PointCloud(a, b, epsilon=eps)
    out = sinkhorn.Sinkhorn()(linear_problem.LinearProblem(geom))
    return float(out.reg_ot_cost)

for i in range(n_t - 1):
    pred = traj[i][:, -1]
    base = ot_cost(snaps[i], snaps[i + 1])          # "do nothing" baseline
    print(f"t={i}->{i+1}: model {ot_cost(pred, snaps[i+1]):.3f}   no-motion baseline {base:.3f}")

NameError: name 'n_t' is not defined

## Visualize learned trajectories

In [6]:
fig, axes = plt.subplots(1, n_t - 1, figsize=(4 * (n_t - 1), 3.6), sharex=True, sharey=True)
for i, ax in enumerate(np.atleast_1d(axes)):
    ax.scatter(snaps[i + 1][:, 0], snaps[i + 1][:, 1], s=2, c="lightgray", label=f"observed t={i+1}")
    for k in range(40):
        p = traj[i][k]
        ax.plot(p[:, 0], p[:, 1], lw=0.7, alpha=0.7, c="C0")
    ax.scatter(traj[i][:, -1, 0], traj[i][:, -1, 1], s=2, c="C3", alpha=0.4, label="predicted")
    ax.set_title(f"t={i} -> {i+1}")
axes[0].legend(markerscale=4, fontsize=8)
plt.tight_layout(); plt.show()

NameError: name 'n_t' is not defined

## Differences from the TorchCFM version
| TorchCFM | This notebook |
|---|---|
| `torch.nn` MLP + `torchdyn.NeuralODE` | `equinox.nn.MLP` + `diffrax.diffeqsolve` |
| global RNG (`torch.randn`) | explicit `jax.random` keys, also passed to the matcher |
| `loss.backward()` / `optimizer.step()` | `eqx.filter_value_and_grad` + `optax`, wrapped in `eqx.filter_jit` |
| `t, xt, ut = fm.sample_location_and_conditional_flow(x0, x1)` | same call plus a PRNG key (see adapter cell) |
| POT for the Wasserstein metric | `ott-jax` Sinkhorn |

For your trajectory model, the same pattern applies with `x0 ~ N(0, I)`, `x1 = s_{t+1}`, and `h_t` as an extra input to `VelocityNet`. Use plain `ConditionalFlowMatcher` there rather than OT-CFM, as discussed above.